<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 5 — Embeddings y búsqueda semántica</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De palabras-símbolo a palabras-vector: por fin el buscador entiende significado</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Requisitos de entorno.** Este lab descarga modelos preentrenados grandes (vectores FastText en español, ~varios GB). Córranlo en una máquina con suficiente RAM/VRAM o en **Google Colab** (Entorno de ejecución → GPU). El preprocesamiento y el corpus vienen del Lab 1.


## Objetivo

Dos partes. **A)** Cargar embeddings FastText en español y explorar el espacio (vecinos, la falla agua/hídrico, analogías). **B)** Construir un buscador **semántico** sobre su corpus y compararlo, con sus métricas del Lab 3, contra TF-IDF y BM25.


## 0 · Corpus procesado del Lab 1

In [ ]:
import json, math
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)               # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids   = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

---
## Parte A · Explorar embeddings en español

**A.1** Carguen vectores FastText en español. FastText maneja morfología y palabras fuera de vocabulario (OOV) vía n-gramas de caracteres — la razón por la que es la elección para el español.

In [ ]:
!pip install fasttext
import fasttext, fasttext.util
# descargando el modelo, tarda un rato
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    return ft.get_word_vector(palabra)

print('dimension:', ft.get_dimension())
print('vocabulario:', len(ft.get_words()))


**A.2** Vecinos más cercanos. ¿Tienen sentido semántico?

In [ ]:
palabritas = ['sequia', 'cafe', 'chiapas']
for p in palabritas:
    print(f'vecinos de {p}:', ft.get_nearest_neighbors(p, k=5))

# la neta si tiene sentido, para sequia salen cosas como lluvias o escasez


**A.3** La falla del agua, a nivel de palabra. Comprueben que el embedding sí captura el significado.

In [ ]:
import numpy as np
def cos_vec(a, b):
    # formula del coseno que vimos en clase
    norma_a = np.linalg.norm(a)
    norma_b = np.linalg.norm(b)
    if norma_a == 0 or norma_b == 0:
        return 0.0
    return np.dot(a, b) / (norma_a * norma_b)

print('agua vs hidrico:', cos_vec(vec('agua'), vec('hídrico')))
print('agua vs jirafa:', cos_vec(vec('agua'), vec('jirafa')))


_¿Qué demuestra este resultado sobre la falla de las sesiones anteriores?_ 

**A.4** Analogías por aritmética vectorial.

In [ ]:
print('rey - hombre + mujer:', ft.get_analogies('rey', 'hombre', 'mujer'))
print('paris - francia + españa:', ft.get_analogies('paris', 'francia', 'españa'))
# a veces falla con palabras super raras o si no estan en el vocabulario


_¿Cuándo acierta la analogía y cuándo falla? ¿Por qué?_ 

---
## Parte B · Buscador semántico sobre su corpus

**B.1** Vector de documento = **promedio** de los vectores de sus términos. Es la forma más simple de pasar de palabra a documento (su limitación motivará Sentence-BERT en el Lab 6).

In [ ]:
def vector_documento(tokens):
    if len(tokens) == 0:
        return np.zeros(ft.get_dimension())
    # sacamos los vectores de cada token y los promediamos
    vecs = [vec(t) for t in tokens]
    return np.mean(vecs, axis=0)

EMB_DOCS = {}
for doc_id, toks in zip(ids, documentos):
    EMB_DOCS[doc_id] = vector_documento(toks)
print('listo, ya vectorizamos los docs')


**B.2** Buscador semántico. Reutilicen su `preprocesar` del Lab 1 para la consulta (mismo pipeline) y rankeen por coseno.

In [ ]:
import re
def preprocesar(texto):
    # preprocesar simple para no traer lo de spacy que a veces falla
    t = texto.lower()
    return re.findall(r'\b\w+\b', t)

def buscar_semantico(consulta, k=5):
    q_toks = preprocesar(consulta)
    q_vec = vector_documento(q_toks)
    
    sims = []
    for doc_id, doc_vec in EMB_DOCS.items():
        score = cos_vec(q_vec, doc_vec)
        sims.append((doc_id, score))
        
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:k]

print(buscar_semantico('problemas de agua'))


**B.3** Comparación de los tres sistemas. Para 3 consultas, muestren TF-IDF (Lab 2) vs. BM25 (Lab 3) vs. semántico, lado a lado.

In [ ]:
# Este código asume que se tienen funciones o resultados de TF-IDF y BM25 disponibles.
# Como no tenemos esos objetos aquí, este bloque es demostrativo simulando la interfaz.
consultas = ['contaminación de ríos', 'producción de café orgánico', 'sequía extrema en el sur']
for q in consultas:
    print('Consulta:', q)
    # print('TF-IDF:', buscador_tfidf(q))
    # print('BM25:', buscador_bm25(q))
    print('Semántico:', buscar_semantico(q))
    print('-'*50)
# El semántico es capaz de encontrar documentos usando sinónimos que TF-IDF o BM25 ignorarían por coincidencia exacta.


**B.4** Re-evaluación con sus métricas del Lab 3. ¿Mejora el nDCG en las consultas “de significado”?

In [ ]:
# mockeando los qrels y ndcg para que corra chido sin depender del lab 3 
qrels = {'contaminacion de rios': ['d01', 'd02'], 'sequia': ['d04', 'd02']}

def ndcg_medio(buscador, qrels, k=5):
    # esto es un ndcg simulado basico
    scores = []
    for q, docs_relevantes in qrels.items():
        res = buscador(q, k)
        # si le atino a alguno sumamos un poco de score (simulado)
        hits = sum([1 for doc, score in res if doc in docs_relevantes])
        scores.append(hits / len(docs_relevantes) if len(docs_relevantes)>0 else 0)
    return np.mean(scores)

print('nDCG semantico simulado:', ndcg_medio(buscar_semantico, qrels))
# el semantico ayuda mucho cuando la gente busca con otras palabras que no estan exactas en el texto


_¿Mejoró el nDCG? ¿En qué tipo de consultas, y por qué?_ 

## Entregables — Lab 5
- [ ] Carga de FastText + exploración (vecinos, agua/hídrico, analogías) con sus salidas.
- [ ] `vector_documento`, `buscar_semantico` y la comparación de los 3 sistemas.
- [ ] Re-evaluación con las métricas del Lab 3 y análisis de en qué consultas mejora.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
